# Regional sales check

Units sold per region from the committed sample in `data/raw/sales.csv`.

In [1]:
import csv
from collections import Counter, defaultdict
from statistics import mean

with open("data/raw/sales.csv", newline="") as f:
    rows = list(csv.DictReader(f))
print("loaded", len(rows), "rows")

loaded 21 rows


In [2]:
# validation: schema, nulls, ranges, duplicates
expected = ["date", "region", "units", "price"]
print("schema ok:", list(rows[0].keys()) == expected)
print("empty values:", sum(1 for r in rows for v in r.values() if v == ""))
print("units range:", min(int(r["units"]) for r in rows), "to", max(int(r["units"]) for r in rows))
print("price range:", min(float(r["price"]) for r in rows), "to", max(float(r["price"]) for r in rows))
seen, clean = set(), []
for r in rows:
    key = tuple(r.values())
    if key not in seen:
        seen.add(key)
        clean.append(r)
print("duplicates dropped:", len(rows) - len(clean))
print("clean rows:", len(clean))

schema ok: True
empty values: 0
units range: 5 to 17
price range: 9.5 to 10.1
duplicates dropped: 1
clean rows: 20


## Assumptions

- Each row is one day's sales for one region.
- An exact duplicate row is a double entry, so it is dropped once.
- `units` and `price` are complete; the validation cell checks this.

In [3]:
totals = defaultdict(int)
for r in clean:
    totals[r["region"]] += int(r["units"])
for region, units in sorted(totals.items()):
    print(region, units)
print("mean price:", round(mean(float(r["price"]) for r in clean), 3))
print("first 5 rows:")
for r in clean[:5]:
    print(r)

east 77
north 62
south 46
west 32
mean price: 9.723
first 5 rows:
{'date': '2026-01-05', 'region': 'north', 'units': '12', 'price': '9.50'}
{'date': '2026-01-06', 'region': 'south', 'units': '8', 'price': '9.50'}
{'date': '2026-01-07', 'region': 'east', 'units': '15', 'price': '9.75'}
{'date': '2026-01-08', 'region': 'west', 'units': '6', 'price': '10.00'}
{'date': '2026-01-09', 'region': 'north', 'units': '11', 'price': '9.50'}


## Conclusion

After dropping the one duplicate row, `east` sells the most units and `west` the fewest. The sample is small, so treat the gap as a pattern to check, not a result.